# Credit Card Fraud – **Mini-Pipeline** (kompakt)

**Ziel:** Schnell lauffähige Base­lines mit minimaler EDA und klarer Ausgabe.

**Schritte:** Daten laden → Klassenverteilung → Split & Skalieren → Modelle (LogReg, RF) → Metriken → optional Speichern/Laden


In [ ]:
# 1) Imports & Konfiguration (schlank)
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

import joblib

RANDOM_STATE = 42
DATA_PATH = "creditcard.csv"
MODEL_PATH = "rf_fraud_model.joblib"


In [ ]:
# 2) Daten laden & kurzer Überblick
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Datendatei nicht gefunden: {DATA_PATH}. Bitte Pfad prüfen.")

df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
print("\n=== Zielverteilung (Counts) ===")
print(df['Class'].value_counts())
print("\n=== Zielverteilung (%) ===")
print(df['Class'].value_counts(normalize=True).round(4))


In [ ]:
# 3) Features/Ziel vorbereiten (einfach, keine Zusatz-Features)
# - 'Time' wird ignoriert; 'Class' ist Ziel.
X = df.drop(columns=['Class', 'Time'], errors='ignore')
y = df['Class']

# Sanity-Check
print("\nX shape:", X.shape, "| y positives:", int(y.sum()))


In [ ]:
# 4) Train/Test Split + Skalierung
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)


In [ ]:
# 5) Modelle (ohne SMOTE) – class_weight='balanced' statt Resampling
models = {}

lr = LogisticRegression(max_iter=200, class_weight='balanced', random_state=RANDOM_STATE)
lr.fit(X_train_s, y_train)
models['logreg'] = (lr, scaler)

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train_s, y_train)
models['rf'] = (rf, scaler)

print("\nTrainierte Modelle:", list(models.keys()))


In [ ]:
# 6) Evaluierung (kompakt)

def evaluate(model, scaler, X_test, y_test, name):
    Xs = scaler.transform(X_test)
    proba = model.predict_proba(Xs)[:, 1]
    preds = (proba >= 0.5).astype(int)

    print(f"\n--- {name.upper()} ---")
    print(classification_report(y_test, preds, digits=4))
    roc = roc_auc_score(y_test, proba)
    ap = average_precision_score(y_test, proba)
    print(f"ROC AUC: {roc:.4f}")
    print(f"PR AUC (Average Precision): {ap:.4f}")

for name, (m, sc) in models.items():
    evaluate(m, sc, X_test, y_test, name)


In [ ]:
# 7) Persistenz (optional, nur RF)
if os.path.exists(MODEL_PATH):
    saved = joblib.load(MODEL_PATH)
    if isinstance(saved, dict) and 'model' in saved:
        print("\nBestehendes Modell gefunden – geladen.")
    else:
        print("\nWarnung: Unerwartetes Format – bestehende Datei wird nicht geladen.")
else:
    joblib.dump({'model': models['rf'][0], 'scaler': models['rf'][1], 'features': list(X.columns)}, MODEL_PATH)
    print("\nRF-Modell gespeichert:", MODEL_PATH)


# Credit Card Fraud – **Ultra‑Kompakt**

Ziel: Minimal lauffähige Pipeline mit wenigen Ausgaben.

Ablauf: Daten laden → Klassenverteilung → Split+Scale → Modelle (LogReg, RF) → Metriken → **3 Plots** (Class‑Count, ROC, PR) → optional Speichern


In [ ]:
# Imports & Settings
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

import joblib

RANDOM_STATE = 42
DATA_PATH = "creditcard.csv"
MODEL_PATH = "rf_fraud_model.joblib"


In [ ]:
# Daten laden & Zielverteilung
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Datendatei nicht gefunden: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
print("\n=== Zielverteilung (Counts) ===")
print(df['Class'].value_counts())
print("\n=== Zielverteilung (%) ===")
print(df['Class'].value_counts(normalize=True).round(4))

# Plot 1: Class‑Count (einfach)
counts = df['Class'].value_counts().sort_index()
plt.figure(figsize=(4,3))
plt.bar(counts.index.astype(str), counts.values)
plt.title('Zielverteilung (Class)')
plt.xlabel('Class'); plt.ylabel('Count')
plt.tight_layout(); plt.show()


In [ ]:
# Features/Ziel, Split, Skalierung
X = df.drop(columns=['Class', 'Time'], errors='ignore')
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)


In [ ]:
# Modelle (ohne Resampling) – class_weight='balanced'
models = {}

lr = LogisticRegression(max_iter=300, class_weight='balanced', random_state=RANDOM_STATE)
lr.fit(X_train_s, y_train)
models['logreg'] = lr

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train_s, y_train)
models['rf'] = rf

print("\nTrainierte Modelle:", list(models.keys()))


In [ ]:
# Evaluierung (Text + ROC/PR nur für RF)
results = {}
for name, m in models.items():
    proba = m.predict_proba(X_test_s)[:,1]
    preds = (proba >= 0.5).astype(int)
    print(f"\n--- {name.upper()} ---")
    print(classification_report(y_test, preds, digits=4))
    roc = roc_auc_score(y_test, proba)
    ap  = average_precision_score(y_test, proba)
    results[name] = {"roc_auc": roc, "ap": ap}
    print(f"ROC AUC: {roc:.4f} | PR AUC (AP): {ap:.4f}")

# Plots 2 & 3: ROC + PR für RF (meist beste Baseline)
proba_rf = models['rf'].predict_proba(X_test_s)[:,1]

# ROC
fpr, tpr, _ = roc_curve(y_test, proba_rf)
plt.figure(figsize=(4,3))
plt.plot(fpr, tpr)
plt.plot([0,1], [0,1], '--', linewidth=0.8)
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('ROC – RandomForest')
plt.tight_layout(); plt.show()

# PR
prec, rec, _ = precision_recall_curve(y_test, proba_rf)
plt.figure(figsize=(4,3))
plt.plot(rec, prec)
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision‑Recall – RandomForest')
plt.tight_layout(); plt.show()


In [ ]:
# Optional: Persistenz (RF)
if os.path.exists(MODEL_PATH):
    try:
        _ = joblib.load(MODEL_PATH)
        print("\nBestehendes Modell gefunden – keine Überschreibung.")
    except Exception:
        joblib.dump({'model': rf, 'scaler': scaler, 'features': list(X.columns)}, MODEL_PATH)
        print("\nRF‑Modell gespeichert:", MODEL_PATH)
else:
    joblib.dump({'model': rf, 'scaler': scaler, 'features': list(X.columns)}, MODEL_PATH)
    print("\nRF‑Modell gespeichert:", MODEL_PATH)


Minimale To‑Dos (optional): Threshold‑Tuning, einfache K‑Fold‑CV, Monitoring.